In [5]:
!pip install transformers datasets

**Uploading the IMDB Dataset**

In [6]:
from google.colab import files
uploaded = files.upload()

Saving IMDB Dataset.csv to IMDB Dataset.csv


**Data Preprocessing**

In [7]:
import pandas as pd
import re

df = pd.read_csv("IMDB Dataset.csv")

# Reduce size for faster training
df = df.sample(3000)

# Clean text
def clean_text(text):
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z ]', '', text)
    text = text.lower()
    return text

df['review'] = df['review'].apply(clean_text)

# Convert labels
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})

df.dropna(inplace=True)

print(df.head())

                                                  review  sentiment
31919  my girlfriend and i are really into cheesy hor...          0
43791  this movie was groundbreaking in the former so...          1
13088  why do i constantly do this to myself i mean r...          0
253    does anyone remember the alternative comedy sh...          0
10253  meryl streep may be the greatest actor working...          1


**Train-Test Split**

In [8]:
from sklearn.model_selection import train_test_split

train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['review'], df['sentiment'], test_size=0.2, random_state=42)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, random_state=42)

**Tokenization**

In [9]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

train_encodings = tokenizer(list(train_texts), truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(list(val_texts), truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(list(test_texts), truncation=True, padding=True, max_length=128)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

**Dataset Class**

In [10]:
import torch

class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.values

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = Dataset(train_encodings, train_labels)
val_dataset = Dataset(val_encodings, val_labels)
test_dataset = Dataset(test_encodings, test_labels)

**Load Model**

In [11]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


**Training Setup**

In [12]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,   # increase later
    logging_steps=10,
    logging_strategy="steps",
    report_to="none"
)

**Trainer + Training**

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

trainer.train()

Step,Training Loss
10,0.701135
20,0.681384
30,0.678650
40,0.606413
50,0.521531
60,0.453022
70,0.451006
80,0.481593
90,0.394381
100,0.368004


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=150, training_loss=0.4832235018412272, metrics={'train_runtime': 63.9934, 'train_samples_per_second': 37.504, 'train_steps_per_second': 2.344, 'total_flos': 157866633216000.0, 'train_loss': 0.4832235018412272, 'epoch': 1.0})

**Full BERT Evaluation Metrics**

In [14]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

predictions = trainer.predict(test_dataset)
preds = predictions.predictions.argmax(axis=1)

accuracy = accuracy_score(test_labels, preds)
precision, recall, f1, _ = precision_recall_fscore_support(test_labels, preds, average='binary')
cm = confusion_matrix(test_labels, preds)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("Confusion Matrix:\n", cm)

Accuracy: 0.8666666666666667
Precision: 0.8627450980392157
Recall: 0.8741721854304636
F1 Score: 0.868421052631579
Confusion Matrix:
 [[128  21]
 [ 19 132]]


**1. Frozen BERT**

In [15]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# Freeze BERT layers
for param in model.bert.parameters():
    param.requires_grad = False

# Reload fresh model
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

# Freeze all BERT layers
for param in model.bert.parameters():
    param.requires_grad = False

# Trainer again
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

# Train
trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
10,0.695681
20,0.696983
30,0.691768
40,0.706982
50,0.694277
60,0.689348
70,0.700511
80,0.691616
90,0.702369
100,0.705207


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=150, training_loss=0.6983277002970377, metrics={'train_runtime': 30.3515, 'train_samples_per_second': 79.073, 'train_steps_per_second': 4.942, 'total_flos': 157866633216000.0, 'train_loss': 0.6983277002970377, 'epoch': 1.0})

**Frozen BERT Evaluation Metrics**

In [16]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

predictions = trainer.predict(test_dataset)
preds = predictions.predictions.argmax(axis=1)

accuracy = accuracy_score(test_labels, preds)
precision, recall, f1, _ = precision_recall_fscore_support(test_labels, preds, average='binary')
cm = confusion_matrix(test_labels, preds)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("Confusion Matrix:\n", cm)

Accuracy: 0.5
Precision: 0.5019305019305019
Recall: 0.8609271523178808
F1 Score: 0.6341463414634146
Confusion Matrix:
 [[ 20 129]
 [ 21 130]]


**2. Train Last 2 Layers**

In [17]:
from transformers import AutoModelForSequenceClassification

# Reload model again
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

# Freeze all layers first
for param in model.bert.parameters():
    param.requires_grad = False

# Unfreeze last 2 layers
for name, param in model.bert.named_parameters():
    if "encoder.layer.10" in name or "encoder.layer.11" in name:
        param.requires_grad = True

# Trainer again
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

# Train
trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
10,0.693580
20,0.684467
30,0.680704
40,0.671404
50,0.655645
60,0.642631
70,0.632371
80,0.640067
90,0.603728
100,0.635421


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=150, training_loss=0.6300436464945475, metrics={'train_runtime': 26.5453, 'train_samples_per_second': 90.412, 'train_steps_per_second': 5.651, 'total_flos': 157866633216000.0, 'train_loss': 0.6300436464945475, 'epoch': 1.0})

**Only Last 2 Layers Trained Evaluation Metrics**

In [18]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

predictions = trainer.predict(test_dataset)
preds = predictions.predictions.argmax(axis=1)

accuracy = accuracy_score(test_labels, preds)
precision, recall, f1, _ = precision_recall_fscore_support(test_labels, preds, average='binary')
cm = confusion_matrix(test_labels, preds)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("Confusion Matrix:\n", cm)

Accuracy: 0.72
Precision: 0.7342657342657343
Recall: 0.695364238410596
F1 Score: 0.7142857142857143
Confusion Matrix:
 [[111  38]
 [ 46 105]]


**Final Comparison**

Three experiments were conducted to evaluate the impact of fine-tuning on BERT performance. The fully fine-tuned BERT model achieved the highest accuracy of 86.67% and an F1 score of 0.87, demonstrating strong and balanced performance across both classes.

In contrast, the frozen BERT model performed poorly with an accuracy of 50%, indicating that without fine-tuning, the model cannot adapt to the dataset. It showed a strong bias toward predicting the positive class, resulting in high recall but very low precision.

The model trained with only the last two layers fine-tuned achieved moderate performance with an accuracy of 72% and an F1 score of 0.71. This shows that partial fine-tuning improves performance compared to freezing all layers, but it is still not as effective as full fine-tuning.

Overall, the experiments clearly demonstrate that full fine-tuning of BERT is essential for achieving optimal performance in text classification tasks.
